# Cross-model grid overview figures

**Added 2026-08-15.** One figure per condition (`metrics`, `likes_only`, `likes_only_noise`), each showing all 6 main-roster models' 7×7 engagement-disparity grids side by side as small multiples — a single clean view instead of scanning 18 separate per-model PNGs.

**Model scope:** the 6 main-roster models only (Gemma-12B, Gemma-E4B, Qwen3-VL-4B, Qwen3-VL-8B, Ministral-3-8B, Ministral-3-14B) — matches the roster decision already made for the thesis body. Pixtral-12B and Mistral Small 3.1 24B are excluded here, consistent with their appendix-only status.

**Color choice:** uses a validated blue↔red diverging palette (`scripts/validate_palette.js` from the dataviz skill), not this project's earlier red/white/green scheme used in `plot_ab_grid` (`e1_utils/e1_analysis_optimized.py`). That scheme was checked and genuinely fails colorblind-safety — the red/green pair's CVD separation for deuteranopia is ΔE 5.1, below the required 8 floor. Blue (`#2a78d6`) / gray (`#f0efec`) / red (`#e34948`) passes every check (worst-case CVD ΔE 21.6). Red still reads as "low/bad" (never chose correct) and blue as "high/good" (always chose correct), preserving the intuitive read without the accessibility problem. Gray at the 50% midpoint marks chance.

**Deliberately no per-cell numeric labels** (unlike the individual per-model grids, which do label every cell) — at 6 models × 49 cells that would be 294 numbers on screen, working against the "clean overview" goal. Exact per-cell numbers are already available in `docs/THESIS.md` Appendix B and each model's own `*_paired_grid.png`; this figure is for pattern-scanning across models, not precise lookup. The 7 diagonal cells (equal engagement, the "competence" cells discussed in §6.1) are outlined, not labeled, for the same reason.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

REPO_ROOT = Path().resolve().parents[0]  # statistical_analysis/ -> repo root
assert (REPO_ROOT / "experiments" / "e1").exists(), f"Unexpected REPO_ROOT: {REPO_ROOT}"

In [ ]:
MODELS = [
    ("Gemma-12B", "gemma4-12b"),
    ("Gemma-E4B", "gemma4-e4b"),
    ("Qwen3-VL-4B", "qwen3-vl-4b"),
    ("Qwen3-VL-8B", "qwen3-vl-8b"),
    ("Ministral-3-8B", "ministral-3-8b"),
    ("Ministral-3-14B", "ministral-3-14b"),
]

CONDITIONS = [
    ("metrics", "e1_results_metrics_paired.json", "Metrics (all reaction types)"),
    ("likes_only", "e1_results_likes_only_paired.json", "Likes only"),
    ("likes_only_noise", "e1_results_likes_only_noise_paired.json", "Likes only (with noise)"),
]

SCALE_LEVELS = [0, 10, 100, 1000, 10000, 100000, 1000000]
SCALE_LABELS = ["0", "10", "100", "1K", "10K", "100K", "1M"]
N = len(SCALE_LEVELS)

# Validated diverging palette (dataviz skill, references/palette.md — blue/red slots from
# the reference categorical palette, re-checked as a pair via validate_palette.js: worst-case
# CVD ΔE 21.6, well clear of the 8 floor). Replaces this project's existing red/white/green
# scheme (plot_ab_grid), which fails the same check at ΔE 5.1.
DIVERGING_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "red_gray_blue", ["#e34948", "#f0efec", "#2a78d6"]
)

In [ ]:
def build_grid(model_dir: str, fname: str) -> np.ndarray:
    """7x7 array of % chose-correct, rows=incorrect_scale, cols=correct_scale -- same
    orientation as plot_ab_grid, so this figure reads consistently with the existing
    per-model grid PNGs already in the thesis."""
    path = REPO_ROOT / "experiments" / "e1" / model_dir / "outputs" / fname
    data = json.loads(path.read_text())
    by_cell = {}
    for r in data:
        key = (r["correct_scale"], r["incorrect_scale"])
        by_cell.setdefault(key, []).append(r["liked_variant"] == "correct")

    grid = np.full((N, N), np.nan)
    for c_idx, c_scale in enumerate(SCALE_LEVELS):
        for i_idx, i_scale in enumerate(SCALE_LEVELS):
            vals = by_cell.get((c_scale, i_scale))
            if vals:
                grid[i_idx, c_idx] = 100 * sum(vals) / len(vals)
    return grid

In [ ]:
def plot_condition_overview(cond_key: str, fname: str, cond_title: str):
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes_flat = axes.flatten()
    im = None

    for ax, (label, model_dir) in zip(axes_flat, MODELS):
        grid = build_grid(model_dir, fname)
        im = ax.imshow(grid, cmap=DIVERGING_CMAP, vmin=0, vmax=100, aspect="auto")
        ax.set_title(label, fontsize=13, fontweight="bold")
        ax.set_xticks(range(N))
        ax.set_yticks(range(N))
        ax.set_xticklabels(SCALE_LABELS, fontsize=8, rotation=45)
        ax.set_yticklabels(SCALE_LABELS, fontsize=8)
        ax.invert_yaxis()
        # Outline the 7 diagonal (equal-engagement, "competence") cells -- no text label,
        # keeps the panel readable at this density (Section 6.1 discusses these cells).
        for k in range(N):
            ax.add_patch(plt.Rectangle((k - 0.5, k - 0.5), 1, 1, fill=False,
                                         edgecolor="black", linewidth=1.3))

    fig.supxlabel("Correct post reactions shown", fontsize=13)
    fig.supylabel("Incorrect post reactions shown", fontsize=13)
    fig.suptitle(f"% chose correct post — {cond_title}", fontsize=16, fontweight="bold", y=1.02)

    cbar = fig.colorbar(im, ax=axes_flat.tolist(), shrink=0.8, pad=0.02)
    cbar.set_label("% chose the correct post  (gray = 50%, chance)", fontsize=11)

    out_dir = REPO_ROOT / "statistical_analysis" / "outputs"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"grid_overview_{cond_key}.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved to {out_path}")

In [ ]:
for cond_key, fname, cond_title in CONDITIONS:
    plot_condition_overview(cond_key, fname, cond_title)